# CENG501 HW3 – SigLIP-style Vision-Language Model

This notebook implements a lightweight CLIP/SigLIP-style model on concatenated MNIST triples. It follows the homework spec: frozen ViT vision encoder, frozen DistilBERT text encoder, learned 128-d projection heads, SigLIP loss, excluded number `123` from training, and evaluation over all 3-digit captions (000–999).

In [20]:
import math
import random

import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from transformers import DistilBertModel, DistilBertTokenizerFast

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


def set_seed(seed: int = 22):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

Using device: cpu


## 1. Data: concatenated MNIST triples
- Build 3-digit images by sampling three MNIST digits, concatenating horizontally, zero-padding top/bottom to 84x84, then resizing to 224x224 for ViT.
- Captions are the English words for each digit, space-separated (e.g., `1-2-3` -> `one hundred twenty three`).
- Training set **excludes** the number `123`.
- Both train and eval operate on grayscale MNIST but duplicated to 3 channels for ViT.

In [21]:
class ConcatMNIST(Dataset):
    """
    Returns:
      image_84: FloatTensor of shape (1, 84, 84) in [0,1]
      caption:  str (how number is read in English)
      number_str: 3-digit string "000".."999"
    """
    def __init__(self, root: str, split: str = "train", length: int = 20000,
                 exclude_123: bool = False, force_number: str | None = None):
        assert split in {"train", "test"}
        self.split = split
        self.length = length
        self.exclude_123 = exclude_123
        self.force_number = force_number  # e.g. "123" for the special test

        self.mnist = datasets.MNIST(root=root, train=(split == "train"), download=True)
        self.to_tensor = transforms.ToTensor()  # (1,28,28)

        # Build digit -> indices for efficient sampling
        self.digit_to_indices = {d: [] for d in range(10)}
        for i in range(len(self.mnist)):
            _, y = self.mnist[i]
            self.digit_to_indices[int(y)].append(i)

        # Pad top/bottom by 28 each: 28x84 -> 84x84  (matches PDF)
        self.pad84 = transforms.Pad((0, 28, 0, 28), fill=0)

    def _number_to_english(self, n: int) -> str:
        # matching examples like "eight hundred ninety two"
        ones = {
            0: "zero", 1: "one", 2: "two", 3: "three", 4: "four",
            5: "five", 6: "six", 7: "seven", 8: "eight", 9: "nine",
            10: "ten", 11: "eleven", 12: "twelve", 13: "thirteen", 14: "fourteen",
            15: "fifteen", 16: "sixteen", 17: "seventeen", 18: "eighteen", 19: "nineteen"
        }
        tens = {20: "twenty", 30: "thirty", 40: "forty", 50: "fifty",
                60: "sixty", 70: "seventy", 80: "eighty", 90: "ninety"}

        def two_digit(x: int) -> str:
            if x < 20:
                return ones[x]
            t = (x // 10) * 10
            u = x % 10
            return tens[t] if u == 0 else f"{tens[t]} {ones[u]}"

        assert 0 <= n <= 999
        if n < 100:
            return two_digit(n)
        h = n // 100
        r = n % 100
        if r == 0:
            return f"{ones[h]} hundred"
        return f"{ones[h]} hundred {two_digit(r)}"

    def _sample_triplet_indices(self) -> tuple[list[int], str]:
        # if force_number is set, sample those digits (e.g. "123")
        if self.force_number is not None:
            assert len(self.force_number) == 3 and self.force_number.isdigit()
            digits = [int(c) for c in self.force_number]
        else:
            digits = [random.randint(0, 9) for _ in range(3)]

        idxs = [random.choice(self.digit_to_indices[d]) for d in digits]
        number_str = f"{digits[0]}{digits[1]}{digits[2]}"

        # enforce "never show 123" in training when exclude_123=True
        if self.exclude_123 and number_str == "123":
            return self._sample_triplet_indices()

        return idxs, number_str

    def __getitem__(self, _idx):
        idxs, number_str = self._sample_triplet_indices()
        imgs = [self.mnist[i][0] for i in idxs]

        # 28x28 -> tensor (1,28,28), concat to (1,28,84)
        x = torch.cat([self.to_tensor(im) for im in imgs], dim=-1)

        # pad to (1,84,84) matches hw3 pdf
        x84 = self.pad84(x)

        caption = self._number_to_english(int(number_str))
        return x84, caption, number_str

    def __len__(self):
        return self.length

## 2. Dataloaders
Configure training/validation lengths and batch sizes. Validation includes all numbers (123 allowed). Adjust lengths if you want larger experiments.

In [22]:
BATCH_SIZE = 128
TRAIN_SAMPLES = 24000
VAL_SAMPLES = 2000

train_ds = ConcatMNIST('data', split='train', length=TRAIN_SAMPLES, exclude_123=True)
val_ds = ConcatMNIST('data', split='test', length=VAL_SAMPLES, exclude_123=False)

# On macos notebooks, multiprocessing workers can fail to pickle
# notebook-defined classes. Use num_workers=0 to avoid the spawn/pickle
# error (AttributeError: Can't get attribute 'ConcatMNIST' ...)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

## 3. Model: frozen encoders + projection heads
- Vision: `vit_tiny_patch16_224` (192-d embedding).
- Text: `distilbert-base-uncased` (768-d).
- Two learnable linear heads map to 128-d, with L2 normalization.
- Only the projection heads train; encoders are frozen.

In [23]:
class VisionEncoder(nn.Module):
    def __init__(self, model_name: str = "vit_tiny_patch16_224"):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x224):
        return self.model(x224)  # (B,192)

class TextEncoder(nn.Module):
    def __init__(self, model_name: str = "distilbert-base-uncased", pooling: str = "cls"):
        super().__init__()
        self.pooling = pooling
        self.tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
        self.model = DistilBertModel.from_pretrained(model_name)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def forward(self, captions: list[str], device: torch.device):
        tokens = self.tokenizer(
            captions, padding=True, truncation=True, return_tensors="pt"
        ).to(device)
        out = self.model(**tokens).last_hidden_state  # (B, T, 768)

        if self.pooling == "cls":
            return out[:, 0]  # (B,768)
        else:
            return out.mean(dim=1)

class SigLipModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = VisionEncoder()
        self.text = TextEncoder(pooling="cls")

        # two linear layers as required
        self.img_proj = nn.Linear(192, 128)
        self.txt_proj = nn.Linear(768, 128)
        
        # SigLIP learnable temperature and bias parameters, you did not explicitly mention this
        # in the pdf, but I think we need to learn them as they are in the equation and what we saw
        # t_prime is log(temperature), initialized so temperature = exp(t_prime) ≈ 10
        # based on the SigLIP paper and classes
        self.t_prime = nn.Parameter(torch.tensor(math.log(11)))
        self.bias = nn.Parameter(torch.tensor(-9.0))  # negative bias helps early training

    def _preprocess_84_to_224(self, x84: torch.Tensor) -> torch.Tensor:
        """
        x84: (B,1,84,84) in [0,1]
        returns: (B,3,224,224)
        """
        if x84.dim() != 4 or x84.shape[1] != 1 or x84.shape[2:] != (84, 84):
            raise ValueError(f"Expected (B,1,84,84) got {tuple(x84.shape)}")
        x3 = x84.repeat(1, 3, 1, 1)  # -> (B,3,84,84)
        x224 = F.interpolate(x3, size=(224, 224), mode="bilinear", align_corners=False)
        return x224

    def encode_image(self, x84: torch.Tensor) -> torch.Tensor:
        x224 = self._preprocess_84_to_224(x84)
        with torch.no_grad():
            img = self.vision(x224)
        return F.normalize(self.img_proj(img), dim=-1)

    def encode_text(self, captions: list[str]) -> torch.Tensor:
        with torch.no_grad():
            txt = self.text(captions, device=next(self.parameters()).device)
        return F.normalize(self.txt_proj(txt), dim=-1)

    def forward(self, x84: torch.Tensor, captions: list[str]):
        return self.encode_image(x84), self.encode_text(captions)
    
    def get_temperature(self) -> torch.Tensor:
        """Returns the actual temperature value (exp of t_prime)"""
        return torch.exp(self.t_prime)


## 4. SigLIP loss
Implementation mirrors Algorithm 1 from "Sigmoid Loss for Language Image Pre-Training" (ICCV 2023).

Key components:
- **Learnable temperature `t`**: Scales the similarity scores (stored as `t_prime = log(t)` to ensure positivity)
- **Learnable bias `b`**: Shifts the similarity scores
- **Pairwise sigmoid loss**: `loss = -mean(log_sigmoid(y_ij * (t * sim_ij + b)))` where `y_ij = +1` for matches, `-1` for non-matches

The temperature and bias are critical for SigLIP to work properly - they allow the model to calibrate the similarity scores for effective gradient flow.

In [24]:
def siglip_loss(img_emb: torch.Tensor, txt_emb: torch.Tensor, 
                t_prime: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    """
    SigLIP pairwise logistic loss with learnable temperature and bias.
    Based on: "Sigmoid Loss for Language Image Pre-Training"
    
    Args:
      img_emb: L2-normalized image embeddings (B, D)
      txt_emb: L2-normalized text embeddings (B, D)
      t_prime: log(temperature) parameter (scalar)
      bias: bias parameter (scalar)
      
    The formula is:
      logits = dot(img_emb, txt_emb.T) * exp(t_prime) + bias
      labels y_ij = +1 if i==j else -1
      loss = -mean(log_sigmoid(y_ij * logits_ij))
    """
    # temperature is exp(t_prime) to ensure positivity
    temperature = torch.exp(t_prime)
    
    # compute scaled logits with bias
    logits = (img_emb @ txt_emb.T) * temperature + bias  # (B,B)
    
    bsz = img_emb.size(0)
    labels = 2 * torch.eye(bsz, device=img_emb.device) - 1  # diagonal +1, rest -1
    
    return -F.logsigmoid(labels * logits).mean()


## 5. Training utilities
Basic training loop logging validation loss; modify epochs or scheduler as needed.

In [25]:
def train_epoch(model, loader, optimizer):
    model.train()  # only projection layers + t_prime/bias train
    total_loss = 0.0

    for x84, captions, _ in loader:
        x84 = x84.to(DEVICE)
        img_emb, txt_emb = model(x84, list(captions))
        # pass learnable temperature and bias to the loss function
        loss = siglip_loss(img_emb, txt_emb, model.t_prime, model.bias)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x84.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0

    for x84, captions, _ in loader:
        x84 = x84.to(DEVICE)
        img_emb, txt_emb = model(x84, list(captions))
        loss = siglip_loss(img_emb, txt_emb, model.t_prime, model.bias)
        total_loss += loss.item() * x84.size(0)

    return total_loss / len(loader.dataset)

def fit(model, train_loader, val_loader, epochs=10, lr=1e-3):
    # train the projection heads + SigLIP's learnable temperature and bias
    # the encoders remain frozen as required by the homework
    params = (
        list(model.img_proj.parameters()) + 
        list(model.txt_proj.parameters()) +
        [model.t_prime, model.bias]  # include SigLIP learnable params
    )
    optimizer = torch.optim.AdamW(params, lr=lr)

    history = []
    for epoch in range(1, epochs + 1):
        tr = train_epoch(model, train_loader, optimizer)
        va = eval_epoch(model, val_loader)
        temp = model.get_temperature().item()
        bias = model.bias.item()
        history.append({"epoch": epoch, "train_loss": tr, "val_loss": va, 
                        "temperature": temp, "bias": bias})
        print(f"Epoch {epoch}: train {tr:.4f} | val {va:.4f} | temp {temp:.2f} | bias {bias:.2f}")
    return history


## 6. Caption bank (000–999)
Pre-compute text embeddings for all possible captions once, to speed evaluation.

In [26]:
def generate_all_captions(dataset_obj):
    """
    Generates all 1000 captions using the exact English logic defined in the dataset class.
    """
    captions = []
    numbers = []
    for i in range(1000):
        s = f"{i:03d}"
        cap = dataset_obj._number_to_english(i)
        captions.append(cap)
        numbers.append(s)
    return numbers, captions

def compute_caption_bank(model, dataset_obj):
    model.eval()
    numbers, captions = generate_all_captions(dataset_obj)
    embeds = []
    
    with torch.no_grad():
        for i in range(0, len(captions), 64):
            batch = captions[i:i+64]
            # model.encode_text handles tokenization and device placement
            emb = model.encode_text(batch)
            embeds.append(emb)
            
    caption_embeds = torch.cat(embeds, dim=0)  # Shape: (1000, 128)
    return numbers, caption_embeds

## 7. Evaluation: top-k retrieval
Given an image, compute similarity against the caption bank, then compute top-1/top-5 accuracy over a dataset. Also report metrics specifically for the unseen number `123`.

In [27]:
@torch.no_grad()
def evaluate_one(model, x84_single, caption_numbers, caption_embeds, k=5):
    """
    x84_single: (1,84,84) or (1,1,84,84)
    returns top-k predicted number strings
    """
    model.eval()
    if x84_single.dim() == 3:
        x84_single = x84_single.unsqueeze(0)  # (1,1,84,84)
    x84_single = x84_single.to(DEVICE)

    img_emb = model.encode_image(x84_single)  # (1,128)
    sims = (img_emb @ caption_embeds.T).squeeze(0)  # (1000,)
    topk_idx = sims.topk(k).indices.cpu().tolist()
    return [caption_numbers[i] for i in topk_idx], [sims[i].item() for i in topk_idx]

@torch.no_grad()
def evaluate_dataset(model, loader, caption_numbers, caption_embeds):
    model.eval()
    num_to_idx = {s: i for i, s in enumerate(caption_numbers)}

    top1 = 0
    top5 = 0
    total = 0

    for x84, _captions, numbers in loader:
        x84 = x84.to(DEVICE)
        img_emb = model.encode_image(x84)  # (B,128)

        sims = img_emb @ caption_embeds.T  # (B,1000)
        topk = sims.topk(5, dim=-1).indices.cpu()  # (B,5)

        for i, num_str in enumerate(numbers):
            total += 1
            true_idx = num_to_idx[num_str]
            if topk[i, 0].item() == true_idx:
                top1 += 1
            if (topk[i] == true_idx).any().item():
                top5 += 1

    return {"top1": top1 / total, "top5": top5 / total, "total": total}

@torch.no_grad()
def evaluate_123(model, caption_numbers, caption_embeds, samples=1000):
    # build a dataset that generates only "123" with random MNIST instances
    ds_123 = ConcatMNIST("data", split="test", length=samples, exclude_123=False, force_number="123")
    loader_123 = DataLoader(ds_123, batch_size=64, shuffle=False, num_workers=0, pin_memory=(DEVICE.type=="cuda"))
    return evaluate_dataset(model, loader_123, caption_numbers, caption_embeds)


## 8. Run training (example)
Uncomment to train. Keep epochs small on GPU; training only updates the projection heads so it is lightweight.

In [28]:
# Initialize
model = SigLipModel().to(DEVICE)
print(f"Model initialized with temperature={model.get_temperature().item():.2f}, bias={model.bias.item():.2f}")

# Task 1: Train (exclude "123")
# Using 10 epochs, 5 seemed not enough after some testing
history = fit(model, train_loader, val_loader, epochs=10, lr=1e-3)

# Task 2: Caption bank (compute once, reuse)
numbers, caption_embeds = compute_caption_bank(model, train_ds)
caption_embeds = caption_embeds.to(DEVICE)

# Report top-1 / top-5 on test set (>=1000)
val_metrics = evaluate_dataset(model, val_loader, numbers, caption_embeds)

# Report top-1 / top-5 for "123" images (unseen in training)
metrics_123 = evaluate_123(model, numbers, caption_embeds, samples=1000)

print("\n" + "=" * 60)
print("--- Final Results ---")
print("=" * 60)
print(f"Test Set:      Top-1: {val_metrics['top1']:.4f}, Top-5: {val_metrics['top5']:.4f} (N={val_metrics['total']})")
print(f"Unseen '123':  Top-1: {metrics_123['top1']:.4f}, Top-5: {metrics_123['top5']:.4f} (N={metrics_123['total']})")
print(f"\nFinal temperature: {model.get_temperature().item():.2f}, bias: {model.bias.item():.2f}")
print("=" * 60)

model.eval()
for j in range(4):
    x84, caption, num_str = val_ds[j]
    preds, scores = evaluate_one(model, x84, numbers, caption_embeds, k=5)

    plt.figure()
    plt.imshow(x84.squeeze(0), cmap="gray")
    plt.title(f"GT: {num_str} | caption: {caption}")
    plt.axis("off")
    plt.show()

    print("Top-5 predicted numbers:", preds)
    print("Top-5 scores:", [round(s, 4) for s in scores])
    print("-" * 60)


Model initialized with temperature=11.00, bias=-9.00
Epoch 1: train 0.0476 | val 0.0467 | temp 10.88 | bias -8.98
Epoch 2: train 0.0457 | val 0.0465 | temp 10.84 | bias -8.97
Epoch 3: train 0.0440 | val 0.0731 | temp 10.85 | bias -8.95


KeyboardInterrupt: 

## 9. Notes
- **SigLIP requires learnable temperature and bias** - Without these, the model fails to learn meaningful representations.
- Increase `TRAIN_SAMPLES`, `EPOCHS`, and tweak LR for better accuracy.
- If you change encoders, update dimensions in `SigLipModel`.
- Cache `caption_embeds` to disk (`torch.save`) for faster re-runs.
- Ensure the training dataloader never yields the number `123`.
- Report top-1/top-5 on ≥1000 test images plus the unseen `123` set as required.
- The temperature typically increases during training while bias becomes less negative.